# B8. Tinh chỉnh siêu tham số

Chọn siêu tham số cho **hồi quy logistic, rừng ngẫu nhiên, LightGBM** bằng kiểm định chéo 5 fold **chỉ trên tập train**, đúng 5 fold của B7 (`ml/data/folds.csv`).
Tập test chưa được chạm. Trọng số lớp giữ theo B7 (LR, LightGBM không trọng số); riêng RF đưa trọng số vào không gian tìm vì B7 đã ghi kết luận chỉ đúng với tham số mặc định.
Đầu ra: `ml/data/tham_so.json` (bộ tham số mỗi mô hình), `ml/data/oof.parquet` (dự đoán ngoài fold của bộ tham số đã chọn, B9 hiệu chỉnh và B10 chọn ngưỡng dùng lại).

## 0. Nguồn đã tra và cách áp dụng

| Nguồn | Nội dung | Áp dụng |
|---|---|---|
| Bergstra, Bengio (2012), *Random Search for Hyper-Parameter Optimization*, JMLR 13 | Chỉ vài siêu tham số thật sự quan trọng, nên tìm ngẫu nhiên hiệu quả hơn lưới; 60 lần thử độc lập thì xác suất có ít nhất một lần lọt top 5% không gian là 1 − 0,95⁶⁰ ≈ 95% | Ngân sách cố định **60 lần thử hợp lệ** cho RF và LightGBM, đặt trước khi chạy |
| Akiba và cộng sự (2019), *Optuna: A Next-generation Hyperparameter Optimization Framework*, KDD; tài liệu Optuna 4.9 (`TPESampler`, `MedianPruner`) | TPE: 10 lần thử đầu ngẫu nhiên, sau đó lấy mẫu ở vùng đã cho điểm tốt; `MedianPruner` dừng lần thử có điểm trung gian kém hơn trung vị các lần trước ở cùng bước | TPE seed 42 (`multivariate=True` vì `num_leaves` và `max_depth` phụ thuộc nhau); điểm trung gian = AUC trung bình các fold đã chấm, dừng sớm lần thử từ fold thứ 2 |
| LightGBM 4.7, *Parameters Tuning* | Cây mọc theo lá nên giới hạn bằng `num_leaves`, giữ `num_leaves` ≤ 2^`max_depth`; `min_data_in_leaf` là tham số quan trọng nhất chống học thuộc, dữ liệu lớn thì hàng trăm đến hàng nghìn; tốc độ học nhỏ + nhiều vòng cho kết quả tốt hơn; lấy mẫu dòng (`bagging`), cột (`feature_fraction`), điều chuẩn `lambda_l2` | Tìm `num_leaves`, `max_depth`, `min_child_samples`, `colsample_bytree`, `subsample`, `reg_lambda`; **loại tổ hợp `num_leaves` > 2^`max_depth`** không huấn luyện; tốc độ học cố định 0,02, số cây do dừng sớm |
| Probst, Wright, Boulesteix (2019), *Hyperparameters and Tuning Strategies for Random Forest*, WIREs DMKD | RF chạy khá với mặc định; tham số đáng tinh chỉnh nhất là số cột thử ở mỗi lần tách (`mtry`), cỡ nút tối thiểu và tỷ lệ mẫu mỗi cây; số cây không phải tham số cần tìm, càng nhiều càng ổn định | Tìm `max_features`, `min_samples_leaf`, `max_samples`; số cây cố định 300. Thêm `class_weight` vì B7 thấy trọng số chỉ có lợi khi lá mọc tới thuần |
| scikit-learn 1.9, `LogisticRegression` | `C` là nghịch đảo độ mạnh điều chuẩn, C nhỏ = điều chuẩn mạnh | LR chỉ một tham số nên dùng lưới `C` = 10⁻⁴ … 10¹, bước nửa bậc (11 giá trị) |
| Cawley, Talbot (2010), *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation*, JMLR 11 | Chọn tham số tối đa một tiêu chí trên mẫu hữu hạn thì điểm của bộ được chọn lạc quan; mức lạc quan có thể bằng chênh lệch thật giữa các thuật toán | AUC CV của bộ tham số thắng **không** là điểm báo cáo cuối; so sánh mô hình làm ở B11 trên test chạm một lần |
| Lời giải Kaggle Home Credit: kozodoi (top 4%, `code_2_modeling.ipynb`); NoxMoon (`README`) | kozodoi: LightGBM tốc độ học 0,005, `num_leaves` 70, `max_depth` 7, `colsample_bytree` 0,8, `subsample` 0,9, dừng sớm 300 vòng **bằng chính fold chấm AUC**. NoxMoon: tối ưu Bayes 50 vòng, cây nông (độ sâu 4–5), `feature_fraction` ≈ 0,3 là quan trọng | Khoảng tìm phủ các giá trị trên. **Không** dừng sớm bằng fold chấm (bài học bản cũ): tách riêng 10% phần huấn luyện của fold làm tập dừng sớm |

Không áp dụng: lưới đầy đủ cho RF/LightGBM (Bergstra, Bengio); tìm `n_estimators` của RF (Probst); thay đổi đặc trưng hay tiền xử lý (cố định từ B6, B7).

In [1]:
import json
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
SEED = 42
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

f = pd.read_parquet(ROOT / "ml" / "data" / "features.parquet").merge(pd.read_csv(ROOT / "ml" / "data" / "folds.csv"), on="SK_ID_CURR")
y = f.pop("TARGET").to_numpy()
fold = f.pop("fold").to_numpy()
ma_ho_so = f.pop("SK_ID_CURR").to_numpy()
X = f
CHU = X.select_dtypes(exclude="number").columns.tolist()
SO = [c for c in X.columns if c not in CHU]
print(f"Train: {len(X):,} hồ sơ, TARGET=1 {y.mean():.4%}; {len(SO)} cột số, {len(CHU)} cột chữ; fold {np.bincount(fold).tolist()}")

Train: 246,008 hồ sơ, TARGET=1 8.0729%; 109 cột số, 15 cột chữ; fold [49202, 49202, 49202, 49201, 49201]


## 1. Luật tìm (đặt trước khi chạy)

- **Metric chọn:** AUC trung bình 5 fold của B7. Bộ thắng = AUC trung bình cao nhất trong các lần thử chạy đủ 5 fold.
- **Tiền xử lý** như B7, fit trên 4 fold huấn luyện của từng vòng. Vì không phụ thuộc siêu tham số, mỗi vòng chỉ fit một lần rồi mọi lần thử dùng lại.
- **LR:** lưới 11 giá trị `C`. **RF, LightGBM:** TPE seed 42, dừng khi đủ **60 lần thử hợp lệ** (chạy đủ hoặc bị dừng sớm vì kém). Lần thử đầu của RF là tham số mặc định B7.
- **Dừng sớm lần thử kém** (`MedianPruner`, 10 lần thử đầu không dừng): sau mỗi fold báo AUC trung bình các fold đã chấm; từ fold thứ 2, kém hơn trung vị các lần thử trước ở cùng fold thì dừng.
  Thứ tự fold giống nhau ở mọi lần thử nên so cùng fold là công bằng.
- **LightGBM:** tốc độ học 0,02, tối đa 10.000 cây. Ở mỗi vòng, phần huấn luyện (4 fold) tách ngẫu nhiên phân tầng 90/10 (seed 42): học trên 90%, **dừng khi AUC trên 10% không tăng sau 100 vòng**;
  fold chấm không tham gia chọn số cây. Tổ hợp `num_leaves` > 2^`max_depth` bị loại ngay, không huấn luyện, không tính vào 60.
  Số cây cho mô hình cuối (B11) = trung bình số cây tốt nhất của 5 vòng ở bộ thắng.

Khoảng tìm (log = lấy mẫu theo thang log):

| Mô hình | Tham số | Khoảng | Mặc định (B7) |
|---|---|---|---|
| LR | `C` | 10⁻⁴ … 10, bước 10^0,5 | 1 |
| RF | `min_samples_leaf` | 1 … 1.000 (log) | 1 |
| RF | `max_features` (tỷ lệ cột) | 0,02 … 0,25 (log) | √248 / 248 ≈ 0,063 |
| RF | `max_samples` (tỷ lệ mẫu mỗi cây) | 0,2 … 1,0 | 1,0 |
| RF | `class_weight` | không / `balanced` | `balanced` (B7) |
| LightGBM | `max_depth` | 3 … 12 | không giới hạn |
| LightGBM | `num_leaves` | 8 … 256 (log), ≤ 2^`max_depth` | 31 |
| LightGBM | `min_child_samples` | 10 … 2.000 (log) | 20 |
| LightGBM | `colsample_bytree` | 0,1 … 1,0 | 1,0 |
| LightGBM | `subsample` (lấy mẫu mỗi vòng) | 0,5 … 1,0 | 1,0 |
| LightGBM | `reg_lambda` | 10⁻³ … 100 (log) | 0 |

`max_features` của RF chặn trên 0,25 (≈ 62 cột, gấp 4 lần mặc định) vì thời gian tăng gần tỷ lệ với số cột thử; RF 300 cây với 0,2 đã mất 50 giây mỗi vòng.

In [2]:
onehot = make_pipeline(SimpleImputer(strategy="constant", fill_value="THIEU"), OneHotEncoder(handle_unknown="ignore"))


def tien_xu_ly(ten):
    if ten == "LR":
        return make_column_transformer((make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), SO),
                                       (onehot, CHU), sparse_threshold=0)
    if ten == "RF":
        return make_column_transformer((onehot, CHU), remainder="passthrough", sparse_threshold=0)
    return make_column_transformer(
        (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, encoded_missing_value=np.nan), CHU),
        remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")


def du_lieu_fold(ten):
    out = []
    for k in range(5):
        tr, va = fold != k, fold == k
        t = tien_xu_ly(ten)
        Xa, Xb = t.fit_transform(X[tr]), t.transform(X[va])
        if ten == "RF":
            Xa, Xb = Xa.astype(np.float32), Xb.astype(np.float32)
        out.append((Xa, y[tr], Xb, y[va]))
    return out


def cham(ten, du_lieu, tao, trial=None, huan_luyen=False):
    """Chạy 5 vòng; trả AUC, PR-AUC từng fold, dự đoán ngoài fold, số cây dừng sớm (LightGBM), AUC trên dữ liệu đã học."""
    kq = {"auc": [], "pr_auc": [], "so_cay": [], "auc_hl": [], "oof": np.empty(len(y))}
    for k, (Xa, ya, Xb, yb) in enumerate(du_lieu):
        m = tao()
        if ten == "LGBM":
            Xa, Xd, ya, yd = train_test_split(Xa, ya, test_size=0.1, stratify=ya, random_state=SEED)
            m.fit(Xa, ya, eval_X=(Xd,), eval_y=(yd,), eval_metric="auc", categorical_feature=CHU,
                  callbacks=[lgb.early_stopping(100, verbose=False)])
            kq["so_cay"].append(m.best_iteration_)
        else:
            m.fit(Xa, ya)
        p = m.predict_proba(Xb)[:, 1]
        kq["oof"][fold == k] = p
        kq["auc"].append(roc_auc_score(yb, p))
        kq["pr_auc"].append(average_precision_score(yb, p))
        if huan_luyen:
            kq["auc_hl"].append(roc_auc_score(ya, m.predict_proba(Xa)[:, 1]))
        if trial is not None:
            trial.report(np.mean(kq["auc"]), k)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return kq


def tim(ten, du_lieu, lay_tham_so, tao, mac_dinh=None, n=60):
    def muc_tieu(trial):
        p = lay_tham_so(trial)
        kq = cham(ten, du_lieu, lambda: tao(p), trial)
        trial.set_user_attr("auc_fold", kq["auc"])
        return np.mean(kq["auc"])

    def du_n(study, _):
        if sum(not t.user_attrs.get("vo_nghia") for t in study.trials if t.state.is_finished()) >= n:
            study.stop()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
    if mac_dinh:
        study.enqueue_trial(mac_dinh)
    t = time.time()
    study.optimize(muc_tieu, callbacks=[du_n])
    bang = study.trials_dataframe(attrs=("number", "value", "params", "state", "user_attrs"))
    bang["vo_nghia"] = bang["user_attrs_vo_nghia"].eq(True) if "user_attrs_vo_nghia" in bang else False
    print(f"{ten}: {len(bang)} lần thử ({bang.vo_nghia.sum()} tổ hợp vô nghĩa bị loại), "
          f"{(bang.state == 'COMPLETE').sum()} chạy đủ 5 fold, {((bang.state == 'PRUNED') & ~bang.vo_nghia).sum()} dừng sớm vì kém; "
          f"{(time.time() - t) / 60:.0f} phút")
    return study, bang

## 2. Hồi quy logistic: lưới `C`

In [3]:
du_lieu = du_lieu_fold("LR")
luoi_c = np.round(10 ** np.arange(-4, 1.01, 0.5), 6)
lr = pd.DataFrame([{"C": c, **{k: np.mean(v) for k, v in cham("LR", du_lieu, lambda: LogisticRegression(C=c, max_iter=3000)).items() if k in ("auc", "pr_auc")}}
                   for c in luoi_c]).set_index("C")
c_tot = lr.auc.idxmax()
print(f"C tốt nhất: {c_tot} (AUC {lr.auc.max():.4f}; C = 1 mặc định: {lr.auc[1.0]:.4f})")
lr

C tốt nhất: 0.01 (AUC 0.7662; C = 1 mặc định: 0.7660)


,auc,pr_auc
C,,
0.0001,0.7595,0.2395
0.0003,0.7636,0.2459
0.0010,0.7652,0.2485
0.0032,0.7659,0.2496
0.0100,0.7662,0.2500
0.0316,0.7662,0.2499
0.1000,0.7661,0.2498
0.3162,0.7661,0.2498
1.0000,0.7660,0.2498


In [4]:
kq_lr = cham("LR", du_lieu, lambda: LogisticRegression(C=c_tot, max_iter=3000), huan_luyen=True)
del du_lieu

## 3. Rừng ngẫu nhiên: TPE 60 lần thử

Lần thử đầu là tham số mặc định B7 (300 cây thay vì 100; B7 dùng 100).

In [5]:
def tham_so_rf(t):
    return {"min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 1000, log=True),
            "max_features": t.suggest_float("max_features", 0.02, 0.25, log=True),
            "max_samples": t.suggest_float("max_samples", 0.2, 1.0),
            "class_weight": t.suggest_categorical("class_weight", [None, "balanced"])}


def tao_rf(p):
    return RandomForestClassifier(n_estimators=300, **p, random_state=SEED, n_jobs=-1)


du_lieu = du_lieu_fold("RF")
st_rf, bang_rf = tim("RF", du_lieu, tham_so_rf, tao_rf,
                     mac_dinh={"min_samples_leaf": 1, "max_features": np.sqrt(248) / 248, "max_samples": 1.0, "class_weight": "balanced"})
bang_rf[bang_rf.state == "COMPLETE"].sort_values("value", ascending=False).drop(columns=["user_attrs_auc_fold", "vo_nghia"], errors="ignore").head(10)

RF: 60 lần thử (0 tổ hợp vô nghĩa bị loại), 46 chạy đủ 5 fold, 14 dừng sớm vì kém; 99 phút


,number,value,params_class_weight,params_max_features,params_max_samples,params_min_samples_leaf,state
53,53,0.7695,balanced,0.1043,0.5319,6,COMPLETE
22,22,0.7695,balanced,0.1215,0.7102,11,COMPLETE
16,16,0.7693,balanced,0.0997,0.6821,7,COMPLETE
35,35,0.7693,balanced,0.0963,0.5686,6,COMPLETE
46,46,0.7693,balanced,0.1002,0.5566,10,COMPLETE
26,26,0.7693,balanced,0.0997,0.4442,9,COMPLETE
41,41,0.7692,balanced,0.0841,0.4596,6,COMPLETE
49,49,0.7691,balanced,0.0922,0.6681,9,COMPLETE
33,33,0.7691,balanced,0.0988,0.4148,7,COMPLETE
29,29,0.7691,balanced,0.1099,0.3491,6,COMPLETE


In [6]:
p_rf = st_rf.best_params
print("Bộ thắng:", p_rf, f"AUC {st_rf.best_value:.4f}; mặc định (lần thử 0): {st_rf.trials[0].value:.4f}")
kq_rf = cham("RF", du_lieu, lambda: tao_rf(p_rf), huan_luyen=True)
del du_lieu

Bộ thắng: {'min_samples_leaf': 6, 'max_features': 0.10431566828269584, 'max_samples': 0.5319261777965816, 'class_weight': 'balanced'} AUC 0.7695; mặc định (lần thử 0): 0.7649


## 4. LightGBM: TPE 60 lần thử hợp lệ, dừng sớm bằng 10% tách riêng

In [7]:
def tham_so_lgbm(t):
    p = {"max_depth": t.suggest_int("max_depth", 3, 12),
         "num_leaves": t.suggest_int("num_leaves", 8, 256, log=True),
         "min_child_samples": t.suggest_int("min_child_samples", 10, 2000, log=True),
         "colsample_bytree": t.suggest_float("colsample_bytree", 0.1, 1.0),
         "subsample": t.suggest_float("subsample", 0.5, 1.0),
         "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 100, log=True)}
    if p["num_leaves"] > 2 ** p["max_depth"]:
        t.set_user_attr("vo_nghia", True)
        raise optuna.TrialPruned()
    return p


def tao_lgbm(p, so_cay=10000):
    return LGBMClassifier(n_estimators=so_cay, learning_rate=0.02, subsample_freq=1, **p, random_state=SEED, n_jobs=-1, verbose=-1)


du_lieu = du_lieu_fold("LGBM")
st_lgbm, bang_lgbm = tim("LGBM", du_lieu, tham_so_lgbm, tao_lgbm)
bang_lgbm[bang_lgbm.state == "COMPLETE"].sort_values("value", ascending=False).drop(columns=["user_attrs_auc_fold", "user_attrs_vo_nghia", "vo_nghia"], errors="ignore").head(10)

LGBM: 69 lần thử (9 tổ hợp vô nghĩa bị loại), 44 chạy đủ 5 fold, 16 dừng sớm vì kém; 59 phút


,number,value,params_colsample_bytree,params_max_depth,params_min_child_samples,params_num_leaves,params_reg_lambda,params_subsample,state
39,39,0.7858,0.2279,11,513,63,0.2281,0.8811,COMPLETE
43,43,0.7856,0.1572,12,938,63,0.0635,0.8167,COMPLETE
42,42,0.7853,0.2643,9,885,34,0.0203,0.8779,COMPLETE
65,65,0.7853,0.1708,10,187,29,0.0016,0.8936,COMPLETE
54,54,0.7852,0.1624,10,605,30,0.0011,0.9731,COMPLETE
52,52,0.7852,0.1134,9,982,26,0.0021,0.9784,COMPLETE
60,60,0.7852,0.3200,7,1783,105,0.0987,0.7629,COMPLETE
45,45,0.7851,0.3535,7,1765,18,0.0028,0.8710,COMPLETE
38,38,0.7851,0.2517,12,597,43,0.1894,0.8632,COMPLETE
44,44,0.7850,0.3587,10,1882,31,0.0066,0.8873,COMPLETE


In [8]:
p_lgbm = st_lgbm.best_params
kq_lgbm = cham("LGBM", du_lieu, lambda: tao_lgbm(p_lgbm), huan_luyen=True)
so_cay = int(round(np.mean(kq_lgbm["so_cay"])))
print("Bộ thắng:", p_lgbm, f"AUC {st_lgbm.best_value:.4f}")
print("Số cây dừng sớm từng vòng:", kq_lgbm["so_cay"], "→ mô hình cuối", so_cay)
del du_lieu

Bộ thắng: {'max_depth': 11, 'num_leaves': 63, 'min_child_samples': 513, 'colsample_bytree': 0.22793022428076254, 'subsample': 0.8810645691556078, 'reg_lambda': 0.22811820202868793} AUC 0.7858
Số cây dừng sớm từng vòng: [969, 1077, 902, 848, 1059] → mô hình cuối 971


## 5. Độ nhạy: các bộ gần đỉnh

Nếu nhiều bộ tham số khác nhau cho AUC sát bộ thắng thì đỉnh phẳng: chọn bộ nào cũng tương đương, và phần hơn của bộ thắng chủ yếu là may mắn khi chọn (Cawley, Talbot 2010).

In [9]:
def do_nhay(bang, ten):
    xong = bang[bang.state == "COMPLETE"].value.sort_values(ascending=False)
    return {"mô hình": ten, "số lần chạy đủ": len(xong), "AUC bộ thắng": xong.iloc[0], "AUC bộ thứ 5": xong.iloc[4],
            "số bộ trong 0,001 của bộ thắng": int((xong >= xong.iloc[0] - 0.001).sum()),
            "số bộ trong 0,002 của bộ thắng": int((xong >= xong.iloc[0] - 0.002).sum())}


pd.DataFrame([{"mô hình": "LR", "số lần chạy đủ": len(lr), "AUC bộ thắng": lr.auc.max(), "AUC bộ thứ 5": lr.auc.sort_values().iloc[-5],
               "số bộ trong 0,001 của bộ thắng": int((lr.auc >= lr.auc.max() - 0.001).sum()),
               "số bộ trong 0,002 của bộ thắng": int((lr.auc >= lr.auc.max() - 0.002).sum())},
              do_nhay(bang_rf, "RF"), do_nhay(bang_lgbm, "LGBM")]).set_index("mô hình")

,số lần chạy đủ,AUC bộ thắng,AUC bộ thứ 5,"số bộ trong 0,001 của bộ thắng","số bộ trong 0,002 của bộ thắng"
mô hình,,,,,
LR,11,0.7662,0.7660,8,9
RF,46,0.7695,0.7693,15,24
LGBM,44,0.7858,0.7852,20,38


## 6. So với mô hình nền B7

AUC từng fold của B7 chép từ output `07_mo_hinh_nen.ipynb` mục 5 (cùng 5 fold).

In [10]:
B7 = {"LR": [0.7664, 0.7670, 0.7667, 0.7651, 0.7650], "RF": [0.7556, 0.7580, 0.7540, 0.7543, 0.7555], "LGBM": [0.7794, 0.7786, 0.7785, 0.7755, 0.7777]}
kq = {"LR": kq_lr, "RF": kq_rf, "LGBM": kq_lgbm}
tong = pd.DataFrame([{"mô hình": t, "AUC B7": np.mean(B7[t]), "AUC B8": np.mean(kq[t]["auc"]), "độ lệch chuẩn B8": np.std(kq[t]["auc"]),
                      "số fold tăng": int((np.array(kq[t]["auc"]) > np.array(B7[t]) + 5e-5).sum()),
                      "PR-AUC B8": np.mean(kq[t]["pr_auc"]), "AUC huấn luyện B8": np.mean(kq[t]["auc_hl"])} for t in kq]).set_index("mô hình")
tong["chênh B8 − B7"] = tong["AUC B8"] - tong["AUC B7"]
tong["chênh huấn luyện − kiểm định"] = tong["AUC huấn luyện B8"] - tong["AUC B8"]
display(tong)
fold_b8 = pd.DataFrame({t: kq[t]["auc"] for t in kq}).rename_axis("fold")
fold_b8["RF − LR"] = fold_b8.RF - fold_b8.LR
fold_b8["LGBM − RF"] = fold_b8.LGBM - fold_b8.RF
fold_b8

,AUC B7,AUC B8,độ lệch chuẩn B8,số fold tăng,PR-AUC B8,AUC huấn luyện B8,chênh B8 − B7,chênh huấn luyện − kiểm định
mô hình,,,,,,,,
LR,0.7660,0.7662,0.0008,4,0.2500,0.7697,0.0002,0.0035
RF,0.7555,0.7695,0.0010,5,0.2514,0.9981,0.0140,0.2285
LGBM,0.7779,0.7858,0.0012,5,0.2780,0.8956,0.0078,0.1099


,LR,RF,LGBM,RF − LR,LGBM − RF
fold,,,,,
0,0.7664,0.7680,0.7858,0.0016,0.0178
1,0.7672,0.7711,0.7866,0.0040,0.0154
2,0.7668,0.7696,0.7875,0.0028,0.0179
3,0.7654,0.7695,0.7844,0.0041,0.0149
4,0.7652,0.7693,0.7844,0.0041,0.0151


AUC so sánh B7 làm tròn 4 chữ số nên "tăng" tính khi chênh > 0,00005. AUC huấn luyện của LightGBM tính trên 90% phần huấn luyện đã học (không gồm 10% dừng sớm).

## 7. Lưu tham số và dự đoán ngoài fold

In [11]:
tham_so = {"LR": {"C": float(c_tot), "max_iter": 3000},
           "RF": {"n_estimators": 300, **p_rf},
           "LGBM": {"learning_rate": 0.02, "subsample_freq": 1, **p_lgbm, "n_estimators": so_cay, "so_cay_tung_vong": kq_lgbm["so_cay"]}}
(ROOT / "ml" / "data" / "tham_so.json").write_text(json.dumps(tham_so, indent=2, ensure_ascii=False))
oof = pd.DataFrame({"SK_ID_CURR": ma_ho_so, "fold": fold, "TARGET": y, **{t: kq[t]["oof"] for t in kq}})
oof.to_parquet(ROOT / "ml" / "data" / "oof.parquet", index=False)
print(json.dumps(tham_so, indent=2, ensure_ascii=False))
print("AUC tính lại từ oof (gộp 5 fold):", {t: round(roc_auc_score(oof.TARGET, oof[t]), 4) for t in kq})

{
  "LR": {
    "C": 0.01,
    "max_iter": 3000
  },
  "RF": {
    "n_estimators": 300,
    "min_samples_leaf": 6,
    "max_features": 0.10431566828269584,
    "max_samples": 0.5319261777965816,
    "class_weight": "balanced"
  },
  "LGBM": {
    "learning_rate": 0.02,
    "subsample_freq": 1,
    "max_depth": 11,
    "num_leaves": 63,
    "min_child_samples": 513,
    "colsample_bytree": 0.22793022428076254,
    "subsample": 0.8810645691556078,
    "reg_lambda": 0.22811820202868793,
    "n_estimators": 971,
    "so_cay_tung_vong": [
      969,
      1077,
      902,
      848,
      1059
    ]
  }
}


AUC tính lại từ oof (gộp 5 fold): {'LR': 0.7662, 'RF': 0.7695, 'LGBM': 0.7858}


## 8. Nhận xét (sau thực nghiệm)

- **LR:** AUC gần như phẳng với mọi `C` ≥ 0,003 (0,7659–0,7662); 8/11 giá trị trong 0,001 của bộ thắng. Bộ thắng `C` = 0,01 hơn mặc định 0,0002, tăng 4/5 fold: tinh chỉnh không đổi gì đáng kể.
  Với 246.008 hồ sơ và 319 cột, dữ liệu đủ nhiều so với số hệ số nên điều chuẩn ít tác dụng; chỉ khi điều chuẩn rất mạnh (`C` ≤ 0,001) AUC mới giảm.
- **RF:** 0,7555 (B7, 100 cây) → 0,7649 (lần thử 0: cùng tham số mặc định, 300 cây) → 0,7695 (bộ thắng). Khoảng 2/3 phần tăng đến từ số cây, 1/3 từ tinh chỉnh.
  Bộ thắng: lá tối thiểu 6, thử 10% số cột (≈ 26 cột) mỗi lần tách, mỗi cây lấy 53% mẫu. Cả 10 bộ đầu đều `balanced`, khớp kết luận B7. AUC huấn luyện vẫn 0,998: rừng vẫn khớp gần hết dữ liệu học,
  nhưng AUC kiểm định tăng ở cả 5 fold. Sau tinh chỉnh RF vượt LR ở cả 5 fold (+0,002 đến +0,004), đảo thứ tự B7.
- **LightGBM:** +0,0078 so với B7, tăng cả 5 fold. Bộ thắng: 63 lá, `min_child_samples` 513, thử 23% số cột mỗi cây, 848–1.077 cây. Mười bộ đầu đều có `colsample_bytree` 0,11–0,36,
  cùng hướng với lời giải NoxMoon (≈ 0,3). LightGBM hơn RF 0,015–0,018 ở cả 5 fold.
- **Đỉnh phẳng:** 38 bộ LightGBM và 24 bộ RF nằm trong 0,002 của bộ thắng. Bộ thắng không khác biệt thật so với các bộ lân cận; AUC CV của nó lạc quan một chút do được chọn là cao nhất trong nhiều bộ (Cawley, Talbot 2010). Thứ tự cuối giữa ba mô hình được xác nhận ở B11 trên test.
- Chênh huấn luyện − kiểm định của LightGBM tăng từ 0,060 (B7) lên 0,110, trong khi AUC kiểm định cũng tăng: chênh này không phải tiêu chí chọn mô hình, AUC kiểm định mới là tiêu chí.